## Tools In Langchain
### How To Create Tools
### How to use built-in tools and toolkits
### How to use chat models to call tools
### How to pass tool outputs to chat models
This @tool decorator is the simplest way to define a custom tool. The decorator uses the function name as the tool name by default, but this can be overridden by passing a string as the first argument. Additionally, the decorator will use the function's docstring as the tool's description - so a docstring MUST be provided. 

In [1]:
## @tool decorator

from langchain_core.tools import tool

@tool
def division(a:int,b:int)->int:
    """Divide 2 number"""
    return a/b

In [2]:
print(division.name)
print(division.description)
print(division.args)

division
Divide 2 number
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [3]:
## async implementation

from langchain_core.tools import tool


@tool
async def amultiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [4]:
from typing import Annotated,List

@tool
def multiply_by_max(
    a: Annotated[int,"A value"],
    b:Annotated[List[int],"list of ints over which to take maximum"]
)->int:
    """Mulitply a by the maximum of b"""
    return a* max(b)

In [5]:
print(multiply_by_max.args)

{'a': {'description': 'A value', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'list of ints over which to take maximum', 'items': {'type': 'integer'}, 'title': 'B', 'type': 'array'}}


## Structured Tool

The StructuredTool.from_function class method provides a bit more configurability than the @tool decorator, without requiring much additional code.


In [6]:
from langchain_core.tools import StructuredTool


def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


async def amultiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

calculator=StructuredTool.from_function(func=multiply,coroutine=amultiply)

print(f"Await: {await calculator.ainvoke({"a": 2, "b": 5})}")   
print(f"Sync: {calculator.invoke({"a": 2, "b": 3})}")

Await: 10
Sync: 6


In [7]:
import requests

# We pretend to be a normal Chrome browser
headers = {
    'User-Agent': 'MyCoolChatBot/1.0 (contact@example.com)' 
}

response = requests.get("https://www.wikipedia.org", headers=headers)
print(response.status_code) 
# This should now print 200!

200


In [8]:
import requests

url = "https://en.wikipedia.org/w/api.php"
params = {
    "action": "query",
    "format": "json",
    "list": "search",
    "srsearch": "LangChain"
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)
print("First 500 chars:\n", response.text[:500])

Status Code: 403
First 500 chars:
 Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.



## Inbuilt Tools
wikipedia Integration

In [9]:
import wikipedia

# ✅ Set user agent directly on wikipedia package
wikipedia.set_user_agent("MyLangChainBot/1.0 (chayansehgal2971@gmail.com)")

from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=100
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(wiki_tool.invoke({"query": "LangChain"}))

C:\Users\asus\AppData\Local\Temp\ipykernel_23112\3281101353.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import WikipediaAPIWrapper


Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


In [10]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b

In [11]:
wiki_tool.name

'wikipedia'

In [12]:
multiply.name

'multiply'

In [13]:
add.name

'add'

In [14]:
tools=[wiki_tool,add,multiply]

In [15]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\asus\\Desktop\\Python_Krish\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=100)),
 StructuredTool(name='add', description='Adds a and b.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001787BD19260>),
 StructuredTool(name='multiply', description='Multiplies a and b.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000001787BD19620>)]

In [16]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("llama-3.3-70b-versatile",model_provider="groq")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001787D2C8AD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001787D2C97F0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

## Bind the tools with LLM

In [17]:
llm_with_tool=llm.bind_tools(tools)
llm_with_tool

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001787D2C8AD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001787D2C97F0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper aro

In [18]:
from langchain_core.messages import HumanMessage
query="What is 2* 3"
messages = [HumanMessage(query)]
response=llm_with_tool.invoke(messages)
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'jjgqnm42c', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 396, 'total_tokens': 415, 'completion_time': 0.056256487, 'completion_tokens_details': None, 'prompt_time': 0.024140987, 'prompt_tokens_details': None, 'queue_time': 0.052104292, 'total_time': 0.080397474}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fe65f-c9af-7850-90db-d5175dd3b3f0-0' tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'jjgqnm42c', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 396, 'output_tokens': 19, 'total_tokens': 415}


In [19]:
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jjgqnm42c', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 396, 'total_tokens': 415, 'completion_time': 0.056256487, 'completion_tokens_details': None, 'prompt_time': 0.024140987, 'prompt_tokens_details': None, 'queue_time': 0.052104292, 'total_time': 0.080397474}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe65f-c9af-7850-90db-d5175dd3b3f0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'jjgqnm42c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 396, 'output_tokens': 19, 'total_tokens': 415})

In [20]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': 'jjgqnm42c',
  'type': 'tool_call'}]

In [21]:
for tool_call in response.tool_calls:
    selected_tool = {"add": add, "multiply": multiply,"wikipedia":wiki_tool,}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    
messages

[HumanMessage(content='What is 2* 3', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='6', name='multiply', tool_call_id='jjgqnm42c')]

In [22]:
llm_with_tool.invoke(messages)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jy5g3vyzn', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 403, 'total_tokens': 422, 'completion_time': 0.037743999, 'completion_tokens_details': None, 'prompt_time': 0.01949285, 'prompt_tokens_details': None, 'queue_time': 0.052651899, 'total_time': 0.057236849}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe65f-caa4-79f1-bfe1-361a4c280cbf-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'jy5g3vyzn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 403, 'output_tokens': 19, 'total_tokens': 422})

In [23]:
from langchain_core.messages import HumanMessage

query = "What is langchain and what is 5*15?"
messages = [HumanMessage(query)]

ai_msg = llm_with_tool.invoke(messages)
ai_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '7p55ck8ng', 'function': {'arguments': '{"query":"langchain"}', 'name': 'wikipedia'}, 'type': 'function'}, {'id': 'kxm6y9set', 'function': {'arguments': '{"a":5,"b":15}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 401, 'total_tokens': 434, 'completion_time': 0.077582653, 'completion_tokens_details': None, 'prompt_time': 0.020649672, 'prompt_tokens_details': None, 'queue_time': 0.0564974, 'total_time': 0.098232325}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe65f-cb3e-7502-b7cd-7bfd790efd78-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'langchain'}, 'id': '7p55ck8ng', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 5, 'b': 15}, 'id': 'kxm6y9set', 'type': 'tool_call'}], inval

In [24]:
from langchain_core.tools import tool

@tool
def division(a:int, b:int)->int:
    """Divides 2 numbers"""
    return a/b

In [25]:
division.name

'division'

In [26]:
tools_full=[wiki_tool,add,multiply,division]

In [27]:
tools_full

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\asus\\Desktop\\Python_Krish\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=100)),
 StructuredTool(name='add', description='Adds a and b.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001787BD19260>),
 StructuredTool(name='multiply', description='Multiplies a and b.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000001787BD19620>),
 StructuredTool(name='division', description='Divides 2 numbers', args_schema=<class 'langchain_core.utils.pydantic.division'>, func=<function division at 0x000001787BCE5260>)]

In [28]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("llama-3.3-70b-versatile",model_provider="groq")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001787D404A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001787D405450>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [29]:
llm_with_all = llm.bind_tools(tools_full)
print([tool.name for tool in tools_full])

['wikipedia', 'add', 'multiply', 'division']


### 📌 STEP 1 — User Asks a Question


In [30]:
from langchain_core.messages import HumanMessage
# User sends a message
query = "What is 10/2 ?"
messages = [HumanMessage(query)]

print("STEP 1 - messages list so far:")
print(messages)

STEP 1 - messages list so far:
[HumanMessage(content='What is 10/2 ?', additional_kwargs={}, response_metadata={})]


### 📌 STEP 2 — LLM Decides Which Tool to Call


In [31]:
# LLM looks at the question and decides it needs the division tool
# IT DOES NOT CALCULATE YET - just decides which tool to use
response = llm_with_all.invoke(messages) # invoke means to call the LLM with the current messages

print("STEP 2 - LLM Response:")
print("Content    :", response.content)      # Empty! LLM hasn't answered yet
print("Tool Calls :", response.tool_calls)   # LLM says "use division tool"

STEP 2 - LLM Response:
Content    : 
Tool Calls : [{'name': 'division', 'args': {'a': 10, 'b': 2}, 'id': 'bwstkxttp', 'type': 'tool_call'}]


### 📌 STEP 3 — Append AI Response to Messages


In [32]:
# ✅ CRITICAL STEP - append the AI's decision to messages
# This keeps track of the conversation history
messages.append(response)

print("STEP 3 - messages list so far:")
for m in messages:
    print(type(m).__name__, "->", m)

STEP 3 - messages list so far:
HumanMessage -> content='What is 10/2 ?' additional_kwargs={} response_metadata={}
AIMessage -> content='' additional_kwargs={'tool_calls': [{'id': 'bwstkxttp', 'function': {'arguments': '{"a":10,"b":2}', 'name': 'division'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 451, 'total_tokens': 470, 'completion_time': 0.067530225, 'completion_tokens_details': None, 'prompt_time': 0.023137974, 'prompt_tokens_details': None, 'queue_time': 0.051733756, 'total_time': 0.090668199}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fe65f-cc5d-7a11-b909-594e63b6c3b1-0' tool_calls=[{'name': 'division', 'args': {'a': 10, 'b': 2}, 'id': 'bwstkxttp', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 451, 'output_tokens': 19, 'total_tokens': 470}

### 📌 STEP 4 — Actually Execute the Tool


In [33]:
# Now we actually RUN the tool with the arguments LLM suggested
# This is pure Python - no LLM involved here

tool_map = {
    "multiply"  : multiply,
    "division"  : division,
    "wikipedia" : wiki_tool
}

for tool_call in response.tool_calls:
    
    # Find the right tool
    tool_name   = tool_call["name"].lower()
    tool_to_use = tool_map[tool_name]
    
    # Execute the tool
    tool_result = tool_to_use.invoke(tool_call)
    
    print("STEP 4 - Tool Execution:")
    print("Tool Used   :", tool_name)
    print("Tool Result :", tool_result)
    
    # Append result to messages
    messages.append(tool_result)
    print("-----------------------------------")
    print("STEP 4 - messages list so far:")
    for m in messages:
        print(type(m).__name__, "->", m)

STEP 4 - Tool Execution:
Tool Used   : division
Tool Result : content='5.0' name='division' tool_call_id='bwstkxttp'
-----------------------------------
STEP 4 - messages list so far:
HumanMessage -> content='What is 10/2 ?' additional_kwargs={} response_metadata={}
AIMessage -> content='' additional_kwargs={'tool_calls': [{'id': 'bwstkxttp', 'function': {'arguments': '{"a":10,"b":2}', 'name': 'division'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 451, 'total_tokens': 470, 'completion_time': 0.067530225, 'completion_tokens_details': None, 'prompt_time': 0.023137974, 'prompt_tokens_details': None, 'queue_time': 0.051733756, 'total_time': 0.090668199}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fe65f-cc5d-7a11-b909-594e63b6c3b1-0' tool_calls=[{'name': 'division', 'args': {'a': 1

### 📌 STEP 5 — LLM Sees Tool Result and Gives Final Answer


In [34]:
# Now LLM has the full conversation:
# Human asked → LLM decided to use tool → Tool returned result
# LLM can now give a proper human-readable answer

final_response = llm_with_all.invoke(messages)

print("STEP 5 - Final Answer:")
print(final_response.content)
print("Tool Calls:", final_response.tool_calls)  # Empty now - no more tools needed

STEP 5 - Final Answer:
The answer is 5.0.
Tool Calls: []
